<a href="https://colab.research.google.com/github/laraatanfoglio/DeepFake-Detection/blob/main/cupid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepFake Detection - Trends and Application for Computer Vision Exam

**Area:** T1, Multimedia Forensics (deepfake detection, AI-generated media identification)

**Starting point assigned:** Affatato, Mandelli, Cannas, Bestagini, Tubaro, *CUPID*, arXiv 2026 (submitted to IEEE)

---

## 1. Context

Deepfakes targeting a specific person (a *Person-of-Interest*, POI: politician, celebrity, executive) enable disinformation and fraud, including real cases in video calls. Because a POI is usually well documented, a detector can be built around genuine footage of that person.

Two families of POI detectors exist:
1. **POI-specific training:** one model per person, which needs a lot of footage of that person.
2. **General identity representation + reference set:** the encoder is trained on real videos of many subjects; the POI is enrolled only at inference with a few pristine videos, and a test video is scored by its similarity to them. CUPID belongs to this family, together with ID-Reveal and POI-Forensics.

## 2. Gaps assigned by the project

1. Exploit **temporal information**, not just single frames.
2. Combine video detection with **fake audio detection**.
3. Extend the method **beyond Person-of-Interest**.
4. Test **true videoconferencing compression** (WebRTC with H.264 and VP8), not only offline recompression.

## 3. CUPID

**CUPID** is a visual-only POI detector that:
- fits a 3D Morphable Model to each frame and unwraps the face into a **UV texture map**, where each face region always sits at the same position across identities, poses and expressions;
- trains a **Masked Autoencoder (ViT)** on these maps from **real videos only** (VoxCeleb2), with reconstruction, multi-layer identity-contrastive and perceptual losses;
- scores a test video by the **maximum cosine similarity** between its embeddings and those of 10 reference videos (10 frames each);
- produces **interpretability maps** showing which facial regions deviate from the POI's identity.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/CUPID_project
%cd /content/drive/MyDrive/CUPID_project
!git clone https://github.com/polimi-ispl/CUPID.git
%cd CUPID

Mounted at /content/drive
/content/drive/MyDrive/CUPID_project
Cloning into 'CUPID'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 53 (delta 5), reused 50 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 12.70 MiB | 13.68 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/drive/MyDrive/CUPID_project/CUPID


## Environment & Dependencies Installation

Create a dedicated Conda environment and install PyTorch alongside the required face detection, 3DMM reconstruction, and contrastive learning libraries:

In [ ]:
!pip install -q timm albumentations pytorch-metric-learning insightface onnxruntime-gpu scikit-learn
!apt-get -qq install -y ffmpeg

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.


In [ ]:
import shutil
import subprocess
import torch
from pathlib import Path

def process_poi_batch_colab(poi_id, video_paths, cupid_model, uv_extractor):
    tmp_dir = Path(f"/content/tmp_{poi_id}")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    results = {}
    for vid_path in video_paths:
        vid_name = Path(vid_path).stem

        # 1. Degradazione WebRTC VP8 in locale (/content)
        vp8_path = tmp_dir / f"{vid_name}_vp8.webm"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(vid_path),
            "-vf", "scale=-2:360,fps=15",
            "-c:v", "libvpx", "-b:v", "150k", "-g", "30",
            "-deadline", "realtime", "-an", str(vp8_path)
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # 2. Estrazione UV Map (10 frame) + Inference CUPID su GPU
        uv_tensor = uv_extractor(str(vp8_path), num_frames=10).cuda() # [10, 3, 224, 224]
        with torch.no_grad():
            tokens, _, _, _ = cupid_model.forward_encoder(uv_tensor, mask_ratio=0.0)
            # Salva tutti i token [10, 197, 192] così hai già pronti anche i patch della bocca per il Modulo B!
            results[vid_name] = tokens.cpu()

    # 3. Salva su Google Drive e libera lo spazio locale di Colab
    torch.save(results, f"{BASE_DRIVE}/embeddings/{poi_id}_vp8.pt")
    shutil.rmtree(tmp_dir)